In [ ]:
!pip install -q streamlit scikit-learn joblib plotly

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
print(os.path.exists('/content/drive/MyDrive/Dataset/co2_emissions.csv'))

True


**MODEL TRAINING**

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# <<< EDIT THIS to your exact CSV path from Cell 1 >>>
DATA_PATH = "/content/drive/MyDrive/Dataset/co2_emissions.csv"

CATEGORICAL_FEATURES = ["make", "vehicle_class", "transmission", "fuel_type"]
MULTICOLLINEAR_COLS = ["fuel_consumption_city", "fuel_consumption_hwy", "fuel_consumption_comb(mpg)"]
OUTLIER_COLS = ["engine_size", "cylinders", "fuel_consumption_comb(l/100km)"]

def cap_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series.clip(lower=lower, upper=upper), lower, upper

# 1. Load & clean
df = pd.read_csv(DATA_PATH, encoding="latin1")
df.drop_duplicates(inplace=True)
df = df.drop("model", axis=1)

X_raw = df.drop("co2_emissions", axis=1)
y = df["co2_emissions"]

# 2. One-hot encode categoricals
preprocessor = ColumnTransformer(
    transformers=[("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES)],
    remainder="passthrough",
)
X_transformed = preprocessor.fit_transform(X_raw)
onehot_names = preprocessor.named_transformers_["onehot"].get_feature_names_out(CATEGORICAL_FEATURES)
remainder_names = [c for c in X_raw.columns if c not in CATEGORICAL_FEATURES]
X = pd.DataFrame(X_transformed, columns=list(onehot_names) + remainder_names)

# 3. Drop multicollinear columns
X_filtered = X.drop(columns=[c for c in MULTICOLLINEAR_COLS if c in X.columns])

# 4. Outlier capping (save bounds)
outlier_bounds = {}
for col in OUTLIER_COLS:
    if col in X_filtered.columns:
        X_filtered[col], lower, upper = cap_outliers_iqr(X_filtered[col])
        outlier_bounds[col] = (float(lower), float(upper))

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_filtered, y, test_size=0.2, random_state=42)

# 6. Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. Log-transform target
y_train_t = np.log1p(y_train)

# 8. Train Random Forest
model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train_t)

# 9. Evaluate
pred = np.expm1(model.predict(X_test_scaled))
print(f"MAE: {mean_absolute_error(y_test, pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, pred)):.2f}")
print(f"R2: {r2_score(y_test, pred):.4f}")

# 10. Save artifacts
joblib.dump(model, "rf_model.joblib")
joblib.dump(preprocessor, "preprocessor.joblib")
joblib.dump(scaler, "scaler.joblib")

metadata = {
    "categorical_features": CATEGORICAL_FEATURES,
    "category_options": {c: sorted(df[c].dropna().unique().tolist()) for c in CATEGORICAL_FEATURES},
    "raw_feature_order": list(X_raw.columns),
    "final_feature_order": list(X_filtered.columns),
    "multicollinear_cols": MULTICOLLINEAR_COLS,
    "outlier_bounds": outlier_bounds,
    "numeric_ranges": {
        "engine_size": (float(df["engine_size"].min()), float(df["engine_size"].max())),
        "cylinders": (int(df["cylinders"].min()), int(df["cylinders"].max())),
        "fuel_consumption_comb(l/100km)": (
            float(df["fuel_consumption_comb(l/100km)"].min()),
            float(df["fuel_consumption_comb(l/100km)"].max()),
        ),
    },
}
joblib.dump(metadata, "metadata.joblib")
print("\n✅ Saved: rf_model.joblib, preprocessor.joblib, scaler.joblib, metadata.joblib")

MAE: 2.23
RMSE: 3.41
R2: 0.9967

✅ Saved: rf_model.joblib, preprocessor.joblib, scaler.joblib, metadata.joblib


In [40]:
import os
os.makedirs(".streamlit", exist_ok=True)

with open(".streamlit/config.toml", "w") as f:
    f.write("""
[theme]
primaryColor="#ffd200"
backgroundColor="#0f2027"
secondaryBackgroundColor="#134e5e"
textColor="#ffffff"
font="sans serif"
""")

**APP.PY**

In [41]:
%%writefile app.py
import joblib
import numpy as np
import pandas as pd
import streamlit as st
import plotly.graph_objects as go
import time

st.set_page_config(page_title="EcoDrive | CO₂ Estimator", page_icon="🌍", layout="wide")

@st.cache_resource
def load_artifacts():
    model = joblib.load("rf_model.joblib")
    preprocessor = joblib.load("preprocessor.joblib")
    scaler = joblib.load("scaler.joblib")
    metadata = joblib.load("metadata.joblib")
    return model, preprocessor, scaler, metadata

model, preprocessor, scaler, metadata = load_artifacts()

cat_options = metadata["category_options"]
raw_feature_order = metadata["raw_feature_order"]
final_feature_order = metadata["final_feature_order"]
multicollinear_cols = metadata["multicollinear_cols"]
outlier_bounds = metadata["outlier_bounds"]
ranges = metadata["numeric_ranges"]

# Header
st.title("🌍 EcoDrive CO₂ Estimator")
st.markdown("Powered by a **Random Forest** model — estimate a vehicle's tailpipe CO₂ output in seconds.")
st.write("")

# Sidebar inputs
with st.sidebar:
    st.header("🔧 Vehicle Configuration")
    make = st.selectbox("Make", cat_options["make"])
    vehicle_class = st.selectbox("Vehicle Class", cat_options["vehicle_class"])
    transmission = st.selectbox("Transmission", cat_options["transmission"])
    fuel_type = st.selectbox("Fuel Type", cat_options["fuel_type"])

    st.divider()

    engine_min, engine_max = round(ranges["engine_size"][0], 1), round(ranges["engine_size"][1], 1)
    st.markdown(f"**⚙️ Engine Size (L)** &nbsp; _(range: {engine_min} – {engine_max})_")
    engine_size = st.slider("Engine Size (L)", engine_min, engine_max,
                             round((engine_min + engine_max) / 2, 1), 0.1,
                             label_visibility="collapsed")

    st.write("")
    cyl_min, cyl_max = int(ranges["cylinders"][0]), int(ranges["cylinders"][1])
    st.markdown(f"**🔩 Cylinders** &nbsp; _(range: {cyl_min} – {cyl_max})_")
    cylinders = st.slider("Cylinders", cyl_min, cyl_max, int((cyl_min + cyl_max) // 2), 1,
                           label_visibility="collapsed")

    st.write("")
    fc_min, fc_max = round(ranges["fuel_consumption_comb(l/100km)"][0], 1), round(ranges["fuel_consumption_comb(l/100km)"][1], 1)
    st.markdown(f"**⛽ Fuel Consumption (L/100km)** &nbsp; _(range: {fc_min} – {fc_max})_")
    fuel_comb = st.slider("Fuel Consumption", fc_min, fc_max,
                           round((fc_min + fc_max) / 2, 1), 0.1,
                           label_visibility="collapsed")

    st.write("")
    predict_clicked = st.button("🚀 Predict Emissions", use_container_width=True, type="primary")


def predict():
    row = {
        "make": make, "vehicle_class": vehicle_class, "engine_size": engine_size,
        "cylinders": cylinders, "transmission": transmission, "fuel_type": fuel_type,
        "fuel_consumption_city": fuel_comb, "fuel_consumption_hwy": fuel_comb,
        "fuel_consumption_comb(l/100km)": fuel_comb,
        "fuel_consumption_comb(mpg)": 235.0 / fuel_comb if fuel_comb else 0.0,
    }
    input_df = pd.DataFrame([row])[raw_feature_order]

    transformed = preprocessor.transform(input_df)
    onehot_names = preprocessor.named_transformers_["onehot"].get_feature_names_out(metadata["categorical_features"])
    remainder_names = [c for c in raw_feature_order if c not in metadata["categorical_features"]]
    X = pd.DataFrame(transformed, columns=list(onehot_names) + remainder_names)
    X = X.drop(columns=[c for c in multicollinear_cols if c in X.columns])
    X = X[final_feature_order]

    for col, (lower, upper) in outlier_bounds.items():
        if col in X.columns:
            X[col] = X[col].clip(lower=lower, upper=upper)

    X_scaled = scaler.transform(X)
    pred_log = model.predict(X_scaled)[0]
    return np.expm1(pred_log)


def make_gauge(value):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        number={"suffix": " g/km", "font": {"size": 44, "color": "white"}},
        gauge={
            "axis": {"range": [0, 400], "tickcolor": "white"},
            "bar": {"color": "#ffd200"},
            "bgcolor": "rgba(0,0,0,0)",
            "borderwidth": 2,
            "bordercolor": "white",
            "steps": [
                {"range": [0, 150], "color": "#2ecc71"},
                {"range": [150, 250], "color": "#f1c40f"},
                {"range": [250, 400], "color": "#e74c3c"},
            ],
        },
    ))
    fig.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        font={"color": "white"},
        height=320,
        margin=dict(l=20, r=20, t=30, b=10),
    )
    return fig


# Main panel
col1, col2 = st.columns([1, 1.2])

with col1:
    st.markdown("### 📋 Selected Configuration")
    st.markdown(f"""
    - 🚘 **{make}** — {vehicle_class}
    - ⚙️ {engine_size} L | {cylinders} cylinders
    - 🔧 {transmission} | ⛽ {fuel_type}
    - 📊 {fuel_comb} L/100km combined
    """)

with col2:
    if predict_clicked:
        with st.spinner("Crunching the numbers..."):
            time.sleep(0.6)
            prediction = predict()
        st.plotly_chart(make_gauge(prediction), use_container_width=True)

        if prediction < 150:
            st.success("🟢 **Low emissions** — this is an efficient vehicle.")
        elif prediction < 250:
            st.warning("🟡 **Moderate emissions** — typical for this vehicle class.")
        else:
            st.error("🔴 **High emissions** — consider more efficient alternatives.")
    else:
        st.info("👈 Configure the vehicle in the sidebar, then click **Predict Emissions**.")

st.divider()
st.caption("Model: Random Forest Regressor · One-Hot Encoding + Standard Scaling + log1p target transform")

Overwriting app.py


In [42]:
!pip install -q pyngrok

In [43]:
from pyngrok import ngrok

# Paste your authtoken here
NGROK_AUTH_TOKEN = "3F8m6rGp1VAldeeclASWPEeJweh_2RfGKet6HBkkHpLbiVpWg"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [44]:
!pkill -f streamlit
!streamlit run app.py &>/content/logs.txt &
import time
time.sleep(5)
!cat /content/logs.txt

In [45]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(f"app is live at: {public_url}")

app is live at: NgrokTunnel: "https://ouch-monotone-employer.ngrok-free.dev" -> "http://localhost:8501"
